In [ ]:
import numpy as np
from pynq import Overlay, allocate
import wave

ol = Overlay("audio_mix.bit")
dma = ol.axi_dma_0

def load_wav(filename):
    with wave.open(filename, 'rb') as f:
        n_channels = f.getnchannels()
        sampwidth = f.getsampwidth()
        n_frames = f.getnframes()
        raw = f.readframes(n_frames)
    
    samples = np.frombuffer(raw, dtype=np.int16)
    
    if n_channels == 2:
        samples = samples[::2]
    
    return samples

def load_and_pack(filenames):
    assert len(filenames) <= 8, "Maximum 8 channels"
    
    # Load all files
    channels = [load_wav(f) for f in filenames]
    
    # Pad all to the same length (longest file)
    max_len = max(len(c) for c in channels)
    padded = [np.pad(c, (0, max_len - len(c))) for c in channels]
    
    buf = np.zeros((max_len, 8), dtype=np.int16)
    for i, ch in enumerate(padded):
        buf[:, i] = ch
    
    return buf.view(np.int32).reshape(max_len * 4)

def stream_wavs(filenames):
    packed = load_and_pack(filenames)
    
    buf = allocate(shape=(len(packed),), dtype=np.int32)
    buf[:] = packed
    
    dma.sendchannel.transfer(buf)
    dma.sendchannel.wait()
    buf.freebuffer()

In [ ]:
stream_wavs([
    "ring_Dink.wav",
    "ring_Drums.wav",
    "ring_Guitar.wav",
    "ring_Lead_Vox.wav",
    "ring_Piano.wav",
    "ring_Ringtone_Bass.wav",
    "ring_Ringtome_Sample.wav",
    "ring_Vocoder.wav"
])